In [ ]:
! pip install pandas sqlalchemy requests dotenv psycopg2-binary

In [ ]:
import os
import pandas as pd
import json
import requests
from datetime import datetime
from sqlalchemy import create_engine
from curses import raw
import time
from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
# CACHE_FILE = "geocoding_cache.json" 
CACHE_DIR = "cache/"
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
POSTGRES_CONN = "postgresql+psycopg2://postgres:@localhost:5432/dengue-propagation"  

In [ ]:
CSV_INPUT_FILE = "../temp/casos_rj.csv" 
df = pd.read_csv(CSV_INPUT_FILE)
df.columns = [c.strip().lower() for c in df.columns]
df

In [ ]:
df['dengue'].value_counts()

In [ ]:
cache = {}
for file in os.listdir(CACHE_DIR):
    file_path = os.path.join(CACHE_DIR, file)
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            file_cache = json.load(f)
            cache.update(file_cache)
            
new_cache_file = os.path.join(CACHE_DIR, time.strftime("%Y%m%d-%H%M%S") + ".json")
with open(new_cache_file, "w", encoding="utf-8") as f:
    json.dump({}, f, ensure_ascii=False, indent=4)

print(len(cache))

In [ ]:
new_cache_file

In [ ]:
def geocode_address(address):
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": GOOGLE_API_KEY
    }

    response = requests.get(base_url, params=params)
    data = response.json()
    print(data)
    return data

In [ ]:
results = []
null_adresses = set()
new_cache = {}
news_calls_count = 0
cache_count = 0
MAX_CALLS = 10000

for i, row in df.iterrows():
    if news_calls_count >= MAX_CALLS:
        print("Reached limit of new API calls.")
        break

    raw_address = row.get("endereço")
    raw_cep = str(row.get("cep de residência", ""))

    if not isinstance(raw_address, str) or raw_address.strip() == "":
        continue

    full_address = f"{raw_address} - RJ, {raw_cep.replace('.', '')}"
    print(f"{i} - {full_address}")

    if raw_address in cache:
        print("Using cached data ", cache[raw_address])
        cache_count += 1
        geocoded = cache[raw_address]
    else:
        geocoded = geocode_address(full_address)
        new_cache[raw_address] = geocoded
        cache[raw_address] = geocoded

        with open(new_cache_file, "w", encoding="utf-8") as f:
            json.dump(new_cache, f, ensure_ascii=False, indent=4)

        news_calls_count += 1
        print("API CALL: ", news_calls_count)
        # time.sleep(0.1)
    
    found_valid = False

    if geocoded.get("results"):
        for result in geocoded["results"]:
            location = result["geometry"]["location"]
            city = None
            neighborhood = None
            state = None

            for comp in result.get("address_components", []):
                types = comp.get("types", [])

                if "administrative_area_level_1" in types:
                    state = comp.get("short_name")  # RJ
                if "administrative_area_level_2" in types:
                    city = comp.get("long_name")
                if "sublocality_level_1" in types:
                    neighborhood = comp.get("long_name")

            if state != "RJ":
                continue

            x = location.get("lng")
            y = location.get("lat")
            
            if city is not None:
                found_valid = True
                print(f"City: {city}, neighborhood: {neighborhood}, X: {x}, Y: {y}")

                dengue_value = row.get("dengue", "").strip().lower()
                if dengue_value == "não detectável" or dengue_value == "nao detectavel":
                    classification = 5
                else:
                    classification = 10

                results.append({
                    "city": city,
                    "neighborhood": neighborhood,
                    "data_notification": pd.to_datetime(row.get("data da coleta"), errors="coerce"),
                    "data_first_symptoms": pd.to_datetime(row.get("data da coleta"), errors="coerce"), 
                    "classification": classification,
                    "x": x,
                    "y": y
                })

                break
    else:
        city, neighborhood, x, y = None, None, None, None


    if city is None or not found_valid:
        null_adresses.add(full_address) 

print(f"Total new API calls: {news_calls_count}, Total cache hits: {cache_count}")

In [ ]:
len(results)

In [ ]:
df_output = pd.DataFrame(results)
print(df_output['neighborhood'].value_counts())

In [ ]:
df_output[df_output['neighborhood'] == 'Guaratiba']

In [ ]:
df_output[df_output['city'].isnull()]

In [ ]:
len(null_adresses)

In [ ]:
news_calls_count = 0

for address in list(null_adresses):
    print(f"Processing null address: {address}")
    geocoded = geocode_address(address)
    new_cache[address] = geocoded
    cache[address] = geocoded

    with open(new_cache_file, "w", encoding="utf-8") as f:
        json.dump(new_cache, f, ensure_ascii=False, indent=4)

    news_calls_count += 1
    print("API CALL: ", news_calls_count)
    # time.sleep(0.1)

    found_valid = False
    if geocoded.get("results"):
        for result in geocoded["results"]:
            location = result["geometry"]["location"]
            city = None
            neighborhood = None
            state = None

            for comp in result.get("address_components", []):
                types = comp.get("types", [])

                if "administrative_area_level_1" in types:
                    state = comp.get("short_name")  # RJ
                if "administrative_area_level_2" in types:
                    city = comp.get("long_name")
                if "sublocality_level_1" in types:
                    neighborhood = comp.get("long_name")

            if state != "RJ":
                continue

            x = location.get("lng")
            y = location.get("lat")

            if city is not None:
                null_adresses.remove(address)
                found_valid = True

                print(
                    f"City: {city}, neighborhood: {neighborhood}, "
                    f"state: {state}, X: {x}, Y: {y}"
                )

                dengue_value = row.get("dengue", "").strip().lower()
                if dengue_value in ("não detectável", "nao detectavel"):
                    classification = 5
                else:
                    classification = 10

                results.append({
                    "city": city,
                    "neighborhood": neighborhood,
                    "data_notification": pd.to_datetime(
                        row.get("data da coleta"), errors="coerce"
                    ),
                    "data_first_symptoms": pd.to_datetime(
                        row.get("data da coleta"), errors="coerce"
                    ),
                    "classification": classification,
                    "x": x,
                    "y": y
                })

                break


print(f"Total new API calls: {news_calls_count}")

In [ ]:

engine = create_engine(POSTGRES_CONN)
df_output.to_sql("cases", engine, if_exists="replace", index=False)
df_output